# Day-1

In [ ]:
import numpy as np
import time

# Function using plain Python loop
def python_loop(arr):
    start = time.time()
    result = [x * 2 for x in arr]  # Just multiply each element by 2
    end = time.time()
    return end - start

# Function using NumPy
def numpy_operation(arr):
    start = time.time()
    result = arr * 2  # Vectorized operation
    end = time.time()
    return end - start

if __name__ == "__main__":
    size = 10_000_000  # 10 million elements
    py_list = list(range(size))
    np_array = np.arange(size)

    time_python = python_loop(py_list)
    time_numpy = numpy_operation(np_array)

    print(f"Python loop time: {time_python:.5f} seconds")
    print(f"NumPy time: {time_numpy:.5f} seconds")
    print(f"NumPy is about {time_python / time_numpy:.2f}x faster than plain Python loop")



Python loop time: 0.44583 seconds
NumPy time: 0.02654 seconds
NumPy is about 16.80x faster than plain Python loop


# Day-2

Colab Demo: Tiny CUDA Example

In [ ]:
import torch

In [ ]:
# 1. CUDA version
print(f"Cuda verion is = {torch.version.cuda}")

# 2. Number of GPUs available
print(f"Number of gpu available =    {torch.cuda.device_count()}")

# 3. Current GPU index
print(f"Current device {torch.cuda.current_device()}")

# 4. Memory info (allocated vs reserved)
print(f"Memory info = {torch.cuda.memory_summary(device=0, abbreviated=True)}")

# 5. Device properties (cores, memory, etc.)
print(f"Device properties {torch.cuda.get_device_properties(0)}")

Cuda verion is = 12.6
Number of gpu available =    1
Current device 0
Memory info = |===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Requested memory      |      0 B   |      0 B   |      0

In [ ]:
import torch

# 6. Is CUDA initialized yet?
print(f"Check isw cuda have been initialized = {torch.cuda.is_initialized()}")

# 7. Total GPU memory (bytes)
print(f"Total GPU memory = {torch.cuda.get_device_properties(0).total_memory}")

# 8. Device capability (major, minor CUDA arch version)
print(f"Capabilities of device = {torch.cuda.get_device_capability(0)}")

# 9. Current device name (again, shorter way)
print(f"Name of device = {torch.cuda.get_device_name(torch.cuda.current_device())}")

# 10. GPU clock rate (kHz)
# print(f"Gpu clock rate is = {torch.cuda.get_device_properties(0).clock_rate}")

Check isw cuda have been initialized = True
Total GPU memory = 15828320256
Capabilities of device = (7, 5)
Name of device = Tesla T4


Do simple addition operation using cuda driver, using c++

In [ ]:
!nvcc add.cu -o add_program

In [ ]:
!./add_program

Result: 0 0 0 0 0 0 0 0 0 0 


Compare performace of cpu and gpu

In [ ]:
!nvcc compare.cu -o compare_program

In [ ]:
!./compare_program

CPU Time: 1265.27 ms
GPU Time: 847.621 ms
Speedup: 1.49273x


In [ ]:
!nvcc render.cu -o render_program

In [ ]:
!./render_program

CPU Render Time: 11.5011 ms
GPU Render Time: 9.92955 ms
CPU[0]=0.75 | GPU[0]=0
CPU[1]=0.75 | GPU[1]=0
CPU[2]=0.75 | GPU[2]=0
CPU[3]=0.75 | GPU[3]=0
CPU[4]=0.75 | GPU[4]=0


# Day-**3**

A utility function for visulize the performance of our functions

In [ ]:
def compare_performance(cpu_time: float, gpu_time: float) -> None:
    """
    Compare CPU and GPU execution times and print a meaningful performance summary.

    Args:
        cpu_time (float): Execution time using NumPy (CPU).
        gpu_time (float): Execution time using CuPy (GPU).
    """
    if cpu_time == gpu_time:
        print("Both NumPy and CuPy performed equally. No speed advantage.")
        return

    faster = "CuPy" if gpu_time < cpu_time else "NumPy"
    slower = "NumPy" if faster == "CuPy" else "CuPy"
    improvement = abs(cpu_time - gpu_time) / max(cpu_time, gpu_time) * 100

    print(f"{faster} outperformed {slower} by {improvement:.2f}% "
          f"({abs(cpu_time - gpu_time):.4f} seconds faster).")

In [ ]:
import numpy as np
import cupy as cp
import time

def cpu_matmul(n):
    A = np.random.rand(n, n)
    B = np.random.rand(n, n)
    start = time.time()
    C = np.dot(A, B)
    return time.time() - start

def gpu_matmul(n):
    A = cp.random.rand(n, n)
    B = cp.random.rand(n, n)
    cp.cuda.Stream.null.synchronize()  # ensure GPU ready
    start = time.time()
    C = cp.dot(A, B)
    cp.cuda.Stream.null.synchronize()  # wait for computation
    return time.time() - start

n = 5_000
cpu_time = cpu_matmul(n)
gpu_time = gpu_matmul(n)
print(f"CPU time: {cpu_time:.4f}s")
print(f"GPU time: {gpu_time:.4f}s")
# print(f"Speedup: {cpu_time/gpu_time:.2f}x")
compare_performance(cpu_time, gpu_time)


CPU time: 6.1182s
GPU time: 1.1723s
CuPy outperformed NumPy by 80.84% (4.9459 seconds faster).


Materix multiplication to compare performance CPU AND GPU

In [ ]:
import numpy as np
import cupy as cp
import time

# Matrix size
N = 100  # Increase this to exaggerate differences

def numpy_multiplication():
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    start = time.time()
    C = np.dot(A, B)
    end = time.time()
    duration = end - start
    print(f"NumPy (CPU) time: {duration:.4f} seconds")
    return duration

def cupy_multiplication(): #
    A = cp.random.rand(N, N, dtype=cp.float32)
    B = cp.random.rand(N, N, dtype=cp.float32)
    cp.cuda.Stream.null.synchronize()  # flush GPU queue
    start = time.time()
    C = cp.dot(A, B)
    cp.cuda.Stream.null.synchronize()  # wait until GPU finishes
    end = time.time()
    duration = end - start
    print(f"CuPy (GPU) time: {duration:.4f} seconds")
    return duration

if __name__ == "__main__":
    cpu_time = numpy_multiplication()
    gpu_time = cupy_multiplication()
    compare_performance(cpu_time, gpu_time)


NumPy (CPU) time: 0.0003 seconds
CuPy (GPU) time: 0.0003 seconds
NumPy outperformed CuPy by 25.54% (0.0001 seconds faster).


Compare performace by calculating the image pixels of both CPU and GPU

In [ ]:
import numpy as np
import cupy as cp
from PIL import Image
import time
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"


# Load a single image
img = Image.open("/content/img.jpg").convert("RGB")
np_img = np.array(img, dtype=np.uint8)

# Stack it N times to simulate a batch of images
BATCH = 150  # try increasing this to see bigger difference
np_batch = np.stack([np_img] * BATCH)

def numpy_image_processing():  # calculate image pixels using numpy
    start = time.time()
    inverted = 255 - np_batch
    end = time.time()
    print(f"NumPy (CPU) batch time: {end - start:.4f} seconds")
    return end - start

def cupy_image_processing():  # calculate it using cupy
    cp_batch = cp.asarray(np_batch)  # transfer all images to GPU
    cp.cuda.Stream.null.synchronize()
    start = time.time()
    inverted = 255 - cp_batch
    cp.cuda.Stream.null.synchronize()
    end = time.time()
    print(f"CuPy (GPU) batch time: {end - start:.4f} seconds")
    return end - start

if __name__ == "__main__":
    cpu_time = numpy_image_processing()
    gpu_time = cupy_image_processing()
    compare_performance(cpu_time, gpu_time)


NumPy (CPU) batch time: 1.0942 seconds
CuPy (GPU) batch time: 0.1738 seconds
CuPy outperformed NumPy by 84.12% (0.9205 seconds faster).


In [ ]:
!pip install pygame

In [ ]:
!pip install moderngl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 5.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import moderngl
import pygame

# Window
pygame.init()
screen = pygame.display.set_mode((800, 600), pygame.OPENGL | pygame.DOUBLEBUF)
ctx = moderngl.create_context()

# Generate random points
N = 100_000
points = np.random.rand(N, 2).astype('f4') * 2 - 1  # coords in [-1,1]

vbo = ctx.buffer(points.tobytes())
prog = ctx.program(vertex_shader="""
    #version 330
    in vec2 in_pos;
    void main() {
        gl_Position = vec4(in_pos, 0.0, 1.0);
        gl_PointSize = 1.0;
    }
""", fragment_shader="""
    #version 330
    out vec4 fragColor;
    void main() {
        fragColor = vec4(1.0, 0.0, 0.0, 1.0); // red
    }
""")

vao = ctx.simple_vertex_array(prog, vbo, 'in_pos')

# Main loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    ctx.clear(0.0, 0.0, 0.0)
    vao.render(mode=moderngl.POINTS)
    pygame.display.flip()

pygame.quit()


Exception: (detect) glXGetCurrentContext: cannot detect OpenGL context

In [ ]:
!nvcc vector_addition.cu -o vector_addition_program

In [ ]:
!./vector_addition_program

CPU Time: 1381.17 ms
GPU Time: 9.37184 ms
GPU is about 147.374x faster than CPU


# **Day-5**

In [ ]:
import torch

In [ ]:
# import platform
# print(platform.platform())

Linux-6.1.123+-x86_64-with-glibc2.35


**cuDNN**

In [ ]:
# 1. Check GPU availability


print("GPU available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("cuDNN enabled:", torch.backends.cudnn.enabled)

GPU available: True
CUDA version: 12.6
cuDNN version: 91002
cuDNN enabled: True


In [ ]:
# This just runs a single convolution on random data.
import torch, time

# Enable cuDNN
torch.backends.cudnn.enabled = True

# Random tensors (like fake images)
x = torch.randn(64, 3, 128, 128).cuda()   # batch=64, 3-channels
conv = torch.nn.Conv2d(3, 16, 3).cuda()   # small conv layer

start = time.time()
y = conv(x)   # cuDNN accelerates this convolution
torch.cuda.synchronize()
print("Output shape:", y.shape)
print("Time:", time.time() - start, "seconds")


Output shape: torch.Size([64, 16, 126, 126])
Time: 1.0618958473205566 seconds


In [ ]:
import torch, time

torch.backends.cudnn.benchmark = True  # Let cuDNN pick fastest algo

x = torch.randn(128, 3, 224, 224).cuda()
conv = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1).cuda()

start = time.time()
for _ in range(100):
    y = conv(x)
torch.cuda.synchronize()
print("Time with cuDNN:", time.time() - start)


Time with cuDNN: 2.650454521179199


In [ ]:
# *************  We can not run it on cloab because it shines only when we have multiple gpuy main pourpose is to comunicate between multiple gpu through Nvlink ******************
import torch.distributed as dist
dist.init_process_group("nccl")

**RAPIDS**

In [ ]:
# 1. cuDF (GPU DataFrame) vs pandas
!pip install cudf-cu11 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import cudf
import pandas as pd
import numpy as np
import time

# Fake dataset
data = np.random.rand(10**8)

# pandas on CPU
t0 = time.time()
pdf = pd.DataFrame({"x": data})
pdf["y"] = pdf["x"]**2 + 2*pdf["x"] + 1
print("pandas:",pdf, time.time() - t0)

# cuDF on GPU
t1 = time.time()
gdf = cudf.DataFrame({"x": data})
gdf["y"] = gdf["x"]**2 + 2*gdf["x"] + 1
print("----------------------------------------------------------------")
print("cuDF:",gdf, time.time() - t1)


pandas:                  x         y
0         0.859281  3.456926
1         0.687291  2.846950
2         0.696325  2.877518
3         0.485115  2.205567
4         0.480604  2.192187
...            ...       ...
99999995  0.597334  2.551477
99999996  0.618717  2.620244
99999997  0.525262  2.326425
99999998  0.881068  3.538415
99999999  0.835246  3.368126

[100000000 rows x 2 columns] 3.2522382736206055
----------------------------------------------------------------
cuDF:                  x         y
0         0.859281  3.456926
1         0.687291  2.846950
2         0.696325  2.877518
3         0.485115  2.205567
4         0.480604  2.192187
...            ...       ...
99999995  0.597334  2.551477
99999996  0.618717  2.620244
99999997  0.525262  2.326425
99999998  0.881068  3.538415
99999999  0.835246  3.368126

[100000000 rows x 2 columns] 1.8336493968963623


In [ ]:
# 2. cuML (GPU KMeans clustering)
from cuml.cluster import KMeans
import cupy as cp

X = cp.random.rand(10_000, 2)  # 10k points
kmeans = KMeans(n_clusters=4)
labels = kmeans.fit_predict(X)

print("Cluster centers:", kmeans.cluster_centers_)
print("Cluster centers:", len(kmeans.cluster_centers_))


Cluster centers: [[0.23859103 0.7524436 ]
 [0.76462547 0.25778529]
 [0.7399736  0.7589714 ]
 [0.26397076 0.25340307]]
Cluster centers: 4


**CV-CUDA**

In [ ]:
!pip install cvcuda

ERROR: Could not find a version that satisfies the requirement cvcuda (from versions: none)
ERROR: No matching distribution found for cvcuda


In [ ]:
import cv2
import cvcuda
import numpy as np
import torch
import time

# Fake image
img = np.random.randint(0, 256, (1080, 1920, 3), dtype=np.uint8)

# ---------- CPU (OpenCV) ----------
start = time.time()
for _ in range(50):
    out_cpu = cv2.resize(img, (640, 480))
print("OpenCV (CPU) time:", time.time() - start, "sec")

# ---------- GPU (CV-CUDA via Torch tensor) ----------
# (H,W,C) -> (N,H,W,C) because CV-CUDA wants batch dimension
torch_tensor = torch.from_numpy(img).cuda().unsqueeze(0)  # shape (1,H,W,C)

# Layout must match shape dims exactly: "NHWC" for (N,H,W,C)
gpu_img = cvcuda.as_tensor(torch_tensor, "NHWC")

start = time.time()
for _ in range(50):
    out_gpu = cvcuda.resize(gpu_img, (1,480,640,3))  # resize expects full NHWC shape
print("CV-CUDA (GPU) time:", time.time() - start, "sec")


ModuleNotFoundError: No module named 'cvcuda'

In [ ]:
print("CUDA version:", torch.version.cuda)


CUDA version: 12.6


In [ ]:
pip install cvcuda-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 5.9 MB/s eta 0:00:00


In [ ]:
!pip install --quiet --extra-index-url=https://pypi.nvidia.com cuopt-cu12==25.8.* nvidia-cuda-runtime-cu12==12.8.*


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 228.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 169.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 162.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 902.2/902.2 kB 193.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.9/437.9 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 180.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 103.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 224.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.9/525.9 MB 23.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 120.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.0/815.0 MB 49.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install nvidia-pip-cu12 --extra-index-url=https://pypi.ngc.nvidia.com
!pip install nvidia-pip-cuopt-cu12 --extra-index-url=https://pypi.ngc.nvidia.com

In [ ]:
# ========================================================
# Install cuOpt (GPU-accelerated optimization library)
# ========================================================


# ========================================================
# Import cuOpt
# ========================================================
from nvidia.cuopt import RoutingSolver

# ========================================================
# 1. Define the distance matrix
#    - depot is node 0
#    - nodes 1..N are customer locations
# ========================================================
distance_matrix = [
    [0, 10, 15, 20],  # distances from depot to all nodes
    [10, 0, 35, 25],  # distances from node 1
    [15, 35, 0, 30],  # from node 2
    [20, 25, 30, 0]   # from node 3
]

# ========================================================
# 2. Create the routing solver
# ========================================================
solver = RoutingSolver(distance_matrix)

# ========================================================
# 3. Set constraints
#    - 1 vehicle
#    - starts at depot (0)
#    - ends at depot (0)
# ========================================================
solver.set_vehicle_count(1)
solver.set_depot(0)

# ========================================================
# 4. Solve the routing problem
# ========================================================
solution = solver.solve()

# ========================================================
# 5. Print the result
# ========================================================
print("Optimal route:", solution['routes'])
print("Total distance:", solution['cost'])


ModuleNotFoundError: No module named 'nvidia.cuopt'